In [214]:
import json
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np


In [215]:
def series_has_team(series_data: dict, team_name="NRG") -> bool:##########################################
    team_name = team_name.lower()

    for game in series_data.get("seriesState", {}).get("games", []):
        for segment in game.get("segments", []):
            for team in segment.get("teams", []):
                name = team.get("name", "").lower()
                if team_name in name:
                    return True
    return False


In [216]:
from datetime import datetime

def extract_series_time(series_data: dict) -> datetime:
    for game in series_data.get("seriesState", {}).get("games", []):
        started = game.get("startedAt")
        if started:
            return datetime.fromisoformat(started.replace("Z", "+00:00"))
    return datetime.min


In [217]:
import json
from pathlib import Path

def get_all_nrg_series(base_dir="."):
    results = []

    for path in Path(base_dir).glob("series_*_raw.json"):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if series_has_team(data, "NRG"):#########################################
            results.append({
                "series_id": path.stem.replace("series_", "").replace("_raw", ""),
                "time": extract_series_time(data),
                "raw": data
            })

    return results


In [218]:
def get_last_5_nrg_matches():
    series = get_all_nrg_series(".")

    series.sort(key=lambda x: x["time"], reverse=True)
    return series[:5]


In [219]:
nrg_last_5 = get_last_5_nrg_matches()

print(f"Found {len(nrg_last_5)} G2 matches\n")

for i, s in enumerate(nrg_last_5, 1):
    print(f"{i}. Series {s['series_id']} | Time: {s['time']}")


Found 5 G2 matches

1. Series 2843071 | Time: 2025-08-31 20:53:43.202000+00:00
2. Series 2843070 | Time: 2025-08-30 22:01:17.399000+00:00
3. Series 2843069 | Time: 2025-08-29 23:21:10.842000+00:00
4. Series 2843067 | Time: 2025-08-24 20:53:28.494000+00:00
5. Series 2843063 | Time: 2025-08-23 00:26:54.433000+00:00


In [220]:
for i, s in enumerate(nrg_last_5, 1):
    print(s['series_id'])

2843071
2843070
2843069
2843067
2843063


#stage1 feature extraction

In [221]:
import isodate

BUY_PHASE_SECONDS = 30  

def avg_round_duration(series_data):
    durations = []

    for game in series_data["seriesState"]["games"]:
        for seg in game["segments"]:
            if seg["type"] == "round" and seg.get("finished"):
                total = isodate.parse_duration(seg["duration"]).total_seconds()
                active = max(0, total - BUY_PHASE_SECONDS)
                durations.append(active)

    return sum(durations) / len(durations) if durations else 0


In [222]:
from datetime import timedelta

UTILITY_KEYWORDS = [
    "snake-bite", "paint-shells", "guided-salvo",
    "shock-dart", "grenade", "molotov"
]

def early_utility_rate(series_data):
    rounds_with_early_utility = 0
    total_rounds = 0

    for game in series_data["seriesState"]["games"]:
        for round_seg in game["segments"]:
            if round_seg["type"] != "round":
                continue

            total_rounds += 1
            round_start = datetime.fromisoformat(
                round_seg["startedAt"].replace("Z", "+00:00")
            )

            early_window = round_start + timedelta(seconds=15)
            found = False

            for team in round_seg.get("teams", []):
                for src in team.get("damageDealtSources", []):
                    weapon = src["source"]["name"].lower()
                    if any(u in weapon for u in UTILITY_KEYWORDS):
                        found = True
                        break

                if found:
                    break

            if found:
                rounds_with_early_utility += 1

    return rounds_with_early_utility / total_rounds if total_rounds else 0


In [223]:
# def avg_first_contact_time(series_data):
#     timings = []

#     for game in series_data["seriesState"]["games"]:
#         for round_seg in game["segments"]:
#             if round_seg["type"] != "round":
#                 continue

#             elapsed = 0
#             found = False

#             for team in round_seg.get("teams", []):
#                 if team.get("damageDealt", 0) > 0:
#                     found = True
#                     break

#             if found:
#                 # conservative proxy: assume first damage at 15–25s
#                 timings.append(20)

#     return sum(timings) / len(timings) if timings else 0

def avg_first_contact_time(series_data):
    timings = []
    
    for game in series_data["seriesState"]["games"]:
        for round_seg in game["segments"]:
            if round_seg["type"] != "round":
                continue
                
            # Look for damage events in chronological order
            damage_events = []
            for team in round_seg.get("teams", []):
                for damage_source in team.get("damageDealtSources", []):
                    if damage_source.get("damageAmount", 0) > 0:
                        # Estimate: first weapon damage ≈ first contact
                        damage_events.append(15.0)  # Conservative
                        break
            
            if damage_events:
                timings.append(min(damage_events))
            else:
                timings.append(30.0)  # Default late contact
    
    return np.mean(timings) if timings else 30.0



In [224]:
def site_hit_frequency(series_data):
    hits = 0
    attack_rounds = 0

    for game in series_data["seriesState"]["games"]:
        for round_seg in game["segments"]:
            if round_seg["type"] != "round":
                continue

            for team in round_seg.get("teams", []):
                if team.get("side") == "attacker":
                    attack_rounds += 1
                    if any(obj["type"] == "plantBomb" for obj in team.get("objectives", [])):
                        hits += 1
                    break

    return hits / attack_rounds if attack_rounds else 0


In [225]:
def retake_vs_hold_rate(series_data):
    retake = 0
    hold = 0

    for game in series_data["seriesState"]["games"]:
        for round_seg in game["segments"]:
            if round_seg["type"] != "round":
                continue

            plant_occurred = any(
                obj["type"] == "plantBomb"
                for t in round_seg.get("teams", [])
                for obj in t.get("objectives", [])
            )

            for team in round_seg.get("teams", []):
                if team.get("side") == "defender" and team.get("won"):
                    if plant_occurred:
                        retake += 1
                    else:
                        hold += 1

    total = retake + hold
    return {
        "retake_rate": retake / total if total else 0,
        "hold_rate": hold / total if total else 0
    }


In [226]:
# features = []

# for s in nrg_last_5:
#     raw = s["raw"]
#     features.append({
#         "series_id": s["series_id"],
#         "avg_round_duration": avg_round_duration(raw),
#         "early_utility_rate": early_utility_rate(raw),
#         "first_contact_time": avg_first_contact_time(raw),
#         "site_hit_freq": site_hit_frequency(raw),
#         **retake_vs_hold_rate(raw)
#     })

# import pandas as pd
# df = pd.DataFrame(features)
# df


In [227]:
import isodate
from datetime import datetime

UTILITY_KEYWORDS = [
    "snake-bite", "paint-shells", "guided-salvo",
    "shock-dart", "grenade", "molotov"
]

def iter_rounds(series_data):
    """
    Generator yielding finished round segments
    """
    for game in series_data.get("seriesState", {}).get("games", []):
        for seg in game.get("segments", []):
            if seg.get("type") == "round" and seg.get("finished"):
                yield seg


In [228]:
def pistol_conversion_rate(series_data):
    rounds = list(iter_rounds(series_data))
    pistol_indices = [0, 12]  # standard halves

    conversions = 0
    total = 0

    for idx in pistol_indices:
        if idx + 1 >= len(rounds):
            continue

        pistol = rounds[idx]
        follow = rounds[idx + 1]

        for team in pistol["teams"]:
            if team.get("won"):
                total += 1
                follow_team = next(
                    t for t in follow["teams"] if t["id"] == team["id"]
                )
                if follow_team.get("won"):
                    conversions += 1

    return conversions / total if total else 0


In [229]:
def mid_round_damage_ratio(series_data):
    ratios = []

    for r in iter_rounds(series_data):
        total_damage = 0
        mid_damage = 0

        for team in r["teams"]:
            total_damage += team.get("damageDealt", 0)

            # proxy: assume mid-round utility contributes here
            for src in team.get("damageDealtSources", []):
                total_damage += src["damageAmount"]
                if any(u in src["source"]["name"].lower() for u in UTILITY_KEYWORDS):
                    mid_damage += src["damageAmount"]

        if total_damage > 0:
            ratios.append(mid_damage / total_damage)

    return sum(ratios) / len(ratios) if ratios else 0


In [230]:
def avg_default_duration(series_data):
    durations = []

    for r in iter_rounds(series_data):
        start = datetime.fromisoformat(r["startedAt"].replace("Z", "+00:00"))

        for team in r["teams"]:
            if team.get("objectives") or team.get("damageDealtSources"):
                # proxy default duration
                durations.append(20)
                break

    return sum(durations) / len(durations) if durations else 0


In [231]:
def trade_efficiency(series_data):
    assists = 0
    kills = 0

    for r in iter_rounds(series_data):
        for team in r["teams"]:
            assists += team.get("killAssistsReceived", 0)
            kills += team.get("kills", 0)

    return assists / kills if kills else 0


In [232]:
def first_blood_participation(series_data):
    supported = 0
    total = 0

    for r in iter_rounds(series_data):
        total += 1
        contributors = sum(
            1 for team in r["teams"] if team.get("damageDealt", 0) > 0
        )
        if contributors >= 2:
            supported += 1

    return supported / total if total else 0


In [233]:
def late_round_win_rate(series_data):
    wins = 0
    total = 0

    for r in iter_rounds(series_data):
        dur = isodate.parse_duration(r["duration"]).total_seconds()

        if dur > 70:
            total += 1
            if any(t.get("won") for t in r["teams"]):
                wins += 1

    return wins / total if total else 0


In [234]:
def utility_damage_share(series_data):
    utility = 0
    total = 0

    for r in iter_rounds(series_data):
        for team in r["teams"]:
            total += team.get("damageDealt", 0)

            for src in team.get("damageDealtSources", []):
                dmg = src["damageAmount"]
                total += dmg
                if any(u in src["source"]["name"].lower() for u in UTILITY_KEYWORDS):
                    utility += dmg

    return utility / total if total else 0


In [235]:
def post_plant_success_rate(series_data):
    wins = 0
    plants = 0

    for r in iter_rounds(series_data):
        planted = any(
            obj["type"] == "plantBomb"
            for t in r["teams"]
            for obj in t.get("objectives", [])
        )

        if planted:
            plants += 1
            if any(t.get("won") for t in r["teams"]):
                wins += 1

    return wins / plants if plants else 0


In [236]:
def defensive_aggression_rate(series_data):
    early = 0
    total = 0

    for r in iter_rounds(series_data):
        for team in r["teams"]:
            if team.get("side") == "defender":
                total += 1
                if team.get("damageDealt", 0) > 0:
                    early += 1
                break

    return early / total if total else 0


In [237]:
def round_collapse_rate(series_data):
    collapse = 0
    total = 0

    for r in iter_rounds(series_data):
        first_kill_team = None

        for team in r["teams"]:
            if team.get("firstKill"):
                first_kill_team = team
                break

        if first_kill_team:
            total += 1
            if not first_kill_team.get("won"):
                collapse += 1

    return collapse / total if total else 0


In [238]:
features = []

for s in nrg_last_5:
    raw = s["raw"]
    features.append({
        "series_id": s["series_id"],
        "series_id": s["series_id"],
        "avg_round_duration": avg_round_duration(raw),
        "early_utility_rate": early_utility_rate(raw),
        "first_contact_time": avg_first_contact_time(raw),
        "site_hit_freq": site_hit_frequency(raw),
        **retake_vs_hold_rate(raw),
        "pistol_conv": pistol_conversion_rate(raw),
        "mid_round_ratio": mid_round_damage_ratio(raw),
        "default_duration": avg_default_duration(raw),
        "trade_eff": trade_efficiency(raw),
        "first_blood_support": first_blood_participation(raw),
        "late_round_win": late_round_win_rate(raw),
        "utility_share": utility_damage_share(raw),
        "post_plant": post_plant_success_rate(raw),
        "def_aggression": defensive_aggression_rate(raw),
        "collapse_rate": round_collapse_rate(raw),
    })


In [239]:
import pandas as pd
df = pd.DataFrame(features)
df


,series_id,avg_round_duration,early_utility_rate,first_contact_time,site_hit_freq,retake_rate,hold_rate,pistol_conv,mid_round_ratio,default_duration,trade_eff,first_blood_support,late_round_win,utility_share,post_plant,def_aggression,collapse_rate
0,2843071,111.477926,0.352941,15.0,0.602941,0.325000,0.675000,1.0,0.009412,20.0,0.420824,1.000000,1.0,0.009232,1.0,1.0,0.264706
1,2843070,126.255824,0.317647,15.0,0.788235,0.540541,0.459459,1.0,0.004036,20.0,0.463248,0.988235,1.0,0.004021,1.0,1.0,0.364706
2,2843069,115.983049,0.508197,15.0,0.704918,0.500000,0.500000,1.0,0.011975,20.0,0.464368,1.000000,1.0,0.011972,1.0,1.0,0.278689
3,2843067,118.336500,0.200000,15.0,0.533333,0.350000,0.650000,1.0,0.002139,20.0,0.320755,1.000000,1.0,0.002247,1.0,1.0,0.133333
4,2843063,102.850224,0.268657,15.0,0.626866,0.500000,0.500000,1.0,0.006008,20.0,0.403888,1.000000,1.0,0.005658,1.0,1.0,0.313433


In [240]:
df.to_csv("NRGteam_strategy_features.csv", index=False)###############################################


## creating dataset using the extracted features in csv file

In [241]:
import pandas as pd

def get_team_feature_summary(csv_path, series_ids):
    df = pd.read_csv(csv_path)

    df["series_id"] = df["series_id"].astype(str)
    series_ids = [str(s) for s in series_ids]
    
    team_df = df[df["series_id"].isin(series_ids)]

    return {
        "tempo_pacing": {
            "avg_round_duration": team_df["avg_round_duration"].mean(),
            "first_contact_time": team_df["first_contact_time"].mean(),
            "default_duration": team_df["default_duration"].mean(),
            "late_round_win": team_df["late_round_win"].mean(),
        },
        "utility_execution": {
            "early_utility_rate": team_df["early_utility_rate"].mean(),
            "utility_share": team_df["utility_share"].mean(),
            "site_hit_freq": team_df["site_hit_freq"].mean(),
            "post_plant": team_df["post_plant"].mean(),
        },
        "coordination": {
            "trade_eff": team_df["trade_eff"].mean(),
            "first_blood_support": team_df["first_blood_support"].mean(),
            "pistol_conv": team_df["pistol_conv"].mean(),
        },
        "defense_risk": {
            "retake_rate": team_df["retake_rate"].mean(),
            "hold_rate": team_df["hold_rate"].mean(),
            "def_aggression": team_df["def_aggression"].mean(),
            "collapse_rate": team_df["collapse_rate"].mean(),
            "mid_round_ratio": team_df["mid_round_ratio"].mean(),
        }
    }


In [242]:
series_ids = [s["series_id"] for s in nrg_last_5]

features_summary = get_team_feature_summary(
    r"NRGteam_strategy_features.csv",#######################################################
    series_ids
)

import pprint
pprint.pprint(features_summary)


{'coordination': {'first_blood_support': np.float64(0.9976470588235294),
                  'pistol_conv': np.float64(1.0),
                  'trade_eff': np.float64(0.41461647606333524)},
 'defense_risk': {'collapse_rate': np.float64(0.270973291690055),
                  'def_aggression': np.float64(1.0),
                  'hold_rate': np.float64(0.5568918918918919),
                  'mid_round_ratio': np.float64(0.006714103468521861),
                  'retake_rate': np.float64(0.44310810810810813)},
 'tempo_pacing': {'avg_round_duration': np.float64(114.98070461218497),
                  'default_duration': np.float64(20.0),
                  'first_contact_time': np.float64(15.0),
                  'late_round_win': np.float64(1.0)},
 'utility_execution': {'early_utility_rate': np.float64(0.3294883346047007),
                       'post_plant': np.float64(1.0),
                       'site_hit_freq': np.float64(0.651258701670049),
                       'utility_share': np.float64

In [243]:
# ---------- NORMALIZATION & LABEL HELPERS ----------

def to_float(v):
    return float(v) if v is not None else None

def clamp(v, lo=0.05, hi=0.95, default=0.5):
    if v is None:
        return default
    try:
        v = float(v)
    except (TypeError, ValueError):
        return default
    return max(lo, min(v, hi))


def pace_bucket(seconds):
    if seconds < 85:
        return "fast"
    elif seconds < 110:
        return "medium"
    else:
        return "slow"

def contact_bucket(seconds):
    if seconds < 15:
        return "early"
    elif seconds < 25:
        return "mid"
    else:
        return "late"

def rate_label(v):
    if v < 0.3:
        return "low"
    elif v < 0.6:
        return "moderate"
    else:
        return "high"

def risk_label(v):
    if v < 0.2:
        return "stable"
    elif v < 0.35:
        return "moderately unstable"
    else:
        return "volatile"

def coordination_label(v):
    if v < 0.35:
        return "loose"
    elif v < 0.55:
        return "average"
    else:
        return "tight"


In [244]:
## quantitative data -> qualitative data

In [245]:
def interpret_features(fs):
    return {
        # 🟦 Tempo & pacing
        "pace": pace_bucket(fs["avg_round_duration"]),
        "first_contact": contact_bucket(fs["first_contact_time"]),
        "default_phase": contact_bucket(fs["default_duration"]),
        "late_round_strength": rate_label(
            clamp(fs["late_round_win"])
        ),

        # 🟩 Utility & execution
        "utility_execution": {
            "early_utility_rate": fs["early_utility_rate"],
            "utility_share": fs["utility_share"],
            "site_hit_freq": fs["site_hit_freq"],
            "post_plant": fs["post_plant"],
        },

        # 🟨 Coordination
        "coordination": {
            "trade_eff": fs["trade_eff"],
            "first_blood_support": fs["first_blood_support"],
            "pistol_conv": fs["pistol_conv"],
        },

        # 🟥 Defense & risk
        "defense_risk": {
            "hold_rate": fs["hold_rate"],
            "retake_rate": fs["retake_rate"],
            "def_aggression": fs["def_aggression"],
            "collapse_rate": fs["collapse_rate"],
            "mid_round_ratio": fs["mid_round_ratio"],
        },
    }


In [246]:
from collections import Counter

Counter(tuple(sorted(f.keys())) for f in features)


Counter({('avg_round_duration',
          'collapse_rate',
          'def_aggression',
          'default_duration',
          'early_utility_rate',
          'first_blood_support',
          'first_contact_time',
          'hold_rate',
          'late_round_win',
          'mid_round_ratio',
          'pistol_conv',
          'post_plant',
          'retake_rate',
          'series_id',
          'site_hit_freq',
          'trade_eff',
          'utility_share'): 5})

In [247]:
# interpreted = interpret_features(features)
interpreted = [interpret_features(f) for f in features]

import pprint
pprint.pprint(interpreted)


[{'coordination': {'first_blood_support': 1.0,
                   'pistol_conv': 1.0,
                   'trade_eff': 0.420824295010846},
  'default_phase': 'mid',
  'defense_risk': {'collapse_rate': 0.2647058823529412,
                   'def_aggression': 1.0,
                   'hold_rate': 0.675,
                   'mid_round_ratio': 0.009412141602543493,
                   'retake_rate': 0.325},
  'first_contact': 'mid',
  'late_round_strength': 'high',
  'pace': 'slow',
  'utility_execution': {'early_utility_rate': 0.35294117647058826,
                        'post_plant': 1.0,
                        'site_hit_freq': 0.6029411764705882,
                        'utility_share': 0.009232260710690183}},
 {'coordination': {'first_blood_support': 0.9882352941176471,
                   'pistol_conv': 1.0,
                   'trade_eff': 0.46324786324786327},
  'default_phase': 'mid',
  'defense_risk': {'collapse_rate': 0.36470588235294116,
                   'def_aggression': 1.0,
    

In [248]:
#storing the results
payload = {
    "team": "MIBR",
    "scope": "last_5_matches",
    "dimension": "team_strategy",
    "interpreted": interpreted
}

with open("NRG_team_strategy_semantic.json", "w", encoding="utf-8") as f:###########################################
    json.dump(payload, f, indent=2)
